# NeuroMorf: облачная установка harness и один бесплатный обзор

Закреплённый исходный commit: `73d5819a0d72efa94d6d1f611710e801eb24bd48`. DSH, Qwen Code, Gemini CLI и OpenCode устанавливаются из официального npm с проверкой хешей. Эта запись — конечный эксперимент, а не круглосуточный сервер. Бесплатная CPU-сессия Colab; GPU, платные ресурсы и ключи не требуются.

Первая ячейка устанавливает и проверяет CLI, затем запускает **реальный DSH на явно обозначенном тестовом ответе без внешней модели**. Вторая делает **ровно одну попытку** реального обращения к документированной бесплатной модели `mimo-v2.5-free` через OpenCode. Только публичный вопрос проекта; ответ не исполняется. Ошибка или исчерпание квоты остаётся ошибкой. Повторный запуск не сбрасывает попытку. Доступность бесплатного предложения не гарантируется.

[Исходный код и PR](https://github.com/Petr111111110000568/neuromorph-agent-os/pull/12). [Описание маршрута](https://opencode.ai/console/guides). Установленный CLI не означает авторизацию в его собственном сервисе или синхронизацию нативных чатов.


In [ ]:
# Выполняется только в бесплатной CPU-сессии Google Colab. Без Drive mount и секретов.
import os, sys, json, hashlib, pathlib, urllib.request, tarfile, subprocess
assert sys.platform == "linux" and os.environ.get("COLAB_RELEASE_TAG"), "Google Colab required"
SOURCE_COMMIT = "73d5819a0d72efa94d6d1f611710e801eb24bd48"
BASE = pathlib.Path("/content/neuromorph-harnesses-" + SOURCE_COMMIT[:12])
BASE.mkdir(exist_ok=True)
env = {"PATH": "/usr/local/bin:/usr/bin:/bin", "LANG": "C.UTF-8",
       "HOME": str(BASE), "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_NOSYSTEM": "1",
       "GIT_CONFIG_GLOBAL": "/dev/null", "COLAB_RELEASE_TAG": "cloud-notebook"}
REPO = BASE / "repo"
def checked(args, cwd=BASE, timeout=120):
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True,
                          capture_output=True, timeout=timeout).stdout.strip()
if not (REPO / ".git").exists():
    REPO.mkdir(exist_ok=True)
    checked(["git", "init", "."], REPO)
    checked(["git", "remote", "add", "origin", "https://github.com/Petr111111110000568/neuromorph-agent-os.git"], REPO)
    checked(["git", "fetch", "--depth=1", "origin", SOURCE_COMMIT], REPO)
    checked(["git", "checkout", "--detach", "FETCH_HEAD"], REPO)
assert checked(["git", "rev-parse", "HEAD"], REPO) == SOURCE_COMMIT
NODE_VERSION = "24.8.0"
NODE_SHA256 = "2598641d188b41793930917f1a99a81c9615856b4205d408a44ab676c1acbb3d"
node_dir = BASE / ("node-v" + NODE_VERSION + "-linux-x64")
if not node_dir.exists():
    archive = BASE / "node.tar.xz"
    url = "https://nodejs.org/dist/v" + NODE_VERSION + "/node-v" + NODE_VERSION + "-linux-x64.tar.xz"
    with urllib.request.urlopen(url, timeout=40) as response:
        raw = response.read(64 * 1024 * 1024 + 1)
    assert len(raw) <= 64 * 1024 * 1024 and hashlib.sha256(raw).hexdigest() == NODE_SHA256
    archive.write_bytes(raw)
    with tarfile.open(archive, "r:xz") as bundle:
        bundle.extractall(BASE, filter="data")
env["PATH"] = str(node_dir / "bin") + ":" + env["PATH"]
assert checked(["node", "--version"]) == "v" + NODE_VERSION
print(json.dumps({"source_commit": SOURCE_COMMIT, "node": NODE_VERSION, "runtime": "Colab CPU", "secrets_used": False}))

def cloud_step(script, *args):
    result = subprocess.run(["python3", script, "--execute", *args], cwd=REPO, env=env,
                            text=True, capture_output=True, timeout=1000)
    log_name = script.rsplit("/", 1)[-1].replace(".py", "") + ("-live" if "--live" in args else "")
    (BASE / (log_name + ".log")).write_text(result.stdout + result.stderr)
    print(log_name, "exit", result.returncode)
    return result.returncode

install_exit = cloud_step("scripts/bootstrap_cloud_harnesses.py", "--output-dir", "runtime/cloud-harnesses")
installation = json.loads((REPO / "runtime/cloud-harnesses/receipt.json").read_text())
print(json.dumps({"status": installation["status"], "packages": [
    {"id": p["id"], "version": p["version"], "status": p["status"], "error": p.get("error")}
    for p in installation["packages"]]}, ensure_ascii=False, indent=2))
assert install_exit == 0, "Read installation receipt; do not silently enable package scripts"
fixture_exit = cloud_step("scripts/run_dsh_free_review.py")
fixture = json.loads((REPO / "runtime/dsh-fixture/receipt.json").read_text())
print("DSH_FIXTURE_RECEIPT=" + json.dumps(fixture, ensure_ascii=False))
assert fixture_exit == 0, "No live request before real DSH protocol passes"


## Один реальный обзор DSH-FREE-REVIEW-001

Текст и квитанция останутся в выводе этой ячейки. Для сохранения самого блокнота используйте «Копировать на Диск». Не публикуйте личные данные, cookies или токены.


In [ ]:
# Один реальный запрос к опубликованной бесплатной модели OpenCode, без ключей и повторов.
assert fixture_exit == 0
live_exit = cloud_step("scripts/run_dsh_free_review.py", "--live")
receipt_path = REPO / "runtime/dsh-live/receipt.json"
if receipt_path.exists():
    live_receipt = json.loads(receipt_path.read_text())
    print("DSH_LIVE_RECEIPT=" + json.dumps(live_receipt, ensure_ascii=False, indent=2))
else:
    print("No receipt: inspect the saved log. Existing attempt is never reset automatically.")
assert live_exit == 0, "Provider rejected/failed or prior attempt exists; no paid fallback or retry"
